# Graphs for Sign2Kannada
This notebook builds two plots from the landmark dataset: a confusion matrix and feature importance.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

LANDMARKS_CSV = Path('data/landmarks.csv')

if not LANDMARKS_CSV.exists():
    raise FileNotFoundError(f'Missing input file: {LANDMARKS_CSV}')

data = np.loadtxt(str(LANDMARKS_CSV), delimiter=',', skiprows=1)
if data.size == 0:
    raise ValueError('data/landmarks.csv has no samples. Run preprocess.py first.')
if data.ndim == 1:
    data = data.reshape(1, -1)
if data.shape[1] != 64:
    raise ValueError(
        f'Invalid landmarks shape: expected 64 columns (label + 63 landmarks), got {data.shape[1]}.'
    )
if np.isnan(data).any():
    raise ValueError('data/landmarks.csv contains invalid numeric values (NaN).')

y = data[:, 0].astype(int)
X = data[:, 1:]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)

## Confusion Matrix
Shows how often each digit is predicted as every other digit.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
disp = ConfusionMatrixDisplay.from_estimator(
    model,
    X_test,
    y_test,
    ax=ax,
    cmap='Blues',
    values_format='d'
)
ax.set_title('Confusion Matrix (Test Set)')
plt.tight_layout()
plt.show()

## Feature Importance (Top 20)
Top features according to the Random Forest model.

In [ ]:
importances = model.feature_importances_
top_k = 20
indices = np.argsort(importances)[-top_k:]
labels = [f'f{idx}' for idx in indices]

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(range(top_k), importances[indices], color='#4C78A8')
ax.set_yticks(range(top_k))
ax.set_yticklabels(labels)
ax.set_xlabel('Importance')
ax.set_title('Top 20 Feature Importances')
plt.tight_layout()
plt.show()